# 02 - Baseline Encoding Model

Loads video stimuli, extracts AlexNet activations, and trains a ridge
regression encoding model to predict fMRI responses per ROI.

**Fix applied:** the Algonauts video folder contains 1102 files (1000
train + 102 held-out test videos with no fMRI labels). This notebook
now explicitly keeps only the first 1000, and validates the shape
before running any regression so a mismatch never fails silently again.

In [1]:
!git clone https://github.com/sossyh/ffa-dnn-ablation.git
%cd ffa-dnn-ablation

Cloning into 'ffa-dnn-ablation'...
remote: Enumerating objects: 84, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 84 (delta 44), reused 53 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (84/84), 26.41 KiB | 6.60 MiB/s, done.
Resolving deltas: 100% (44/44), done.
/content/ffa-dnn-ablation


In [2]:
!pip install nilearn decord python-dotenv --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 95.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.


## Load dataset link from Colab Secrets

Requires a secret named `DROPBOX_DATASET_LINK` set via the key icon in
the left sidebar, with notebook access turned on.

In [3]:
import os
from google.colab import userdata

os.environ["DROPBOX_DATASET_LINK"] = userdata.get("DROPBOX_DATASET_LINK")
print("Link loaded:", os.environ["DROPBOX_DATASET_LINK"] is not None)

Link loaded: True


In [4]:
import sys
sys.path.append(".")

from src.data_loading import (
    download_algonauts_data,
    load_all_rois,
    get_video_list,
    load_video_frames,
)
from src.models.alexnet_wrapper import AlexNetWrapper
from src.encoding_model import train_and_evaluate_encoding_model, region_mean_accuracy

In [5]:
download_algonauts_data(data_dir="data/raw")

Download complete.


## Load fMRI ROI data

In [6]:
fmri_dir = "data/raw/participants_data_v2021/mini_track"
subject = "sub01"

roi_data = load_all_rois(fmri_dir, subject)

for roi_name, data in roi_data.items():
    print(f"{roi_name}: {data.shape}")

NUM_TRAIN_VIDEOS = roi_data["FFA"].shape[0]
print(f"\nNumber of training videos (from fMRI data): {NUM_TRAIN_VIDEOS}")

V1: (1000, 232)
V2: (1000, 231)
V3: (1000, 261)
V4: (1000, 107)
LOC: (1000, 1843)
EBA: (1000, 351)
FFA: (1000, 68)
STS: (1000, 341)
PPA: (1000, 425)

Number of training videos (from fMRI data): 1000


## Load video list

The video folder contains 1102 files: 1000 training videos with fMRI
labels, plus 102 held-out test videos from the Algonauts challenge that
have no fMRI data here. We keep only the first `NUM_TRAIN_VIDEOS`
(matched to the fMRI data itself, not hardcoded), so this stays correct
even if you switch ROI files or subjects later.

In [7]:
video_dir = "data/raw/AlgonautsVideos268_All_30fpsmax"
video_paths = get_video_list(video_dir)

print(f"Total video files found: {len(video_paths)}")

video_paths = video_paths[:NUM_TRAIN_VIDEOS]

assert len(video_paths) == NUM_TRAIN_VIDEOS, (
    f"Video count ({len(video_paths)}) does not match fMRI row count "
    f"({NUM_TRAIN_VIDEOS}). Check the video folder contents before continuing."
)

print(f"Using {len(video_paths)} training videos (matched to fMRI data)")
print("First 3 paths:", video_paths[:3])
print("Last 3 paths:", video_paths[-3:])

Total video files found: 1102
Using 1000 training videos (matched to fMRI data)
First 3 paths: ['data/raw/AlgonautsVideos268_All_30fpsmax/0001_0-0-1-6-7-2-8-0-17500167280.mp4', 'data/raw/AlgonautsVideos268_All_30fpsmax/0002_0-0-4-3146384004.mp4', 'data/raw/AlgonautsVideos268_All_30fpsmax/0003_0-0-8-1-2-4-0-0-3500812400.mp4']
Last 3 paths: ['data/raw/AlgonautsVideos268_All_30fpsmax/0998_wc-8lYUtCY5Er7d_31.mp4', 'data/raw/AlgonautsVideos268_All_30fpsmax/0999_wc-QqJnzWt2uQtA_34.mp4', 'data/raw/AlgonautsVideos268_All_30fpsmax/1000_yt_R-1887PalKvzM_35.mp4']


## Extract AlexNet activations for every video

Cached to disk so this expensive step never needs to be recomputed
unless the video count changes (in which case the cache is
automatically invalidated and rebuilt, so the mismatch from before
can't happen silently again).

In [8]:
import numpy as np
from tqdm import tqdm

os.makedirs("data/processed", exist_ok=True)
activation_cache_path = "data/processed/alexnet_activations.npz"

need_to_extract = True

if os.path.exists(activation_cache_path):
    print("Found cached activations, checking they match current video count...")
    cached = np.load(activation_cache_path, allow_pickle=True)
    all_activations = cached["all_activations"].item()
    cached_num_videos = next(iter(all_activations.values())).shape[0]

    if cached_num_videos == len(video_paths):
        print(f"Cache matches ({cached_num_videos} videos). Using cached activations.")
        need_to_extract = False
    else:
        print(
            f"Cache has {cached_num_videos} videos but current list has "
            f"{len(video_paths)}. Cache is stale, re-extracting."
        )

if need_to_extract:
    print("Extracting activations (this will take a while)...")
    alexnet = AlexNetWrapper()

    all_activations = {layer: [] for layer in alexnet.layers}

    for video_path in tqdm(video_paths):
        frames = load_video_frames(video_path, num_frames=8)
        layer_acts = alexnet.extract(frames)
        for layer in alexnet.layers:
            all_activations[layer].append(layer_acts[layer])

    for layer in all_activations:
        all_activations[layer] = np.stack(all_activations[layer])

    np.savez(activation_cache_path, all_activations=all_activations)
    print("Saved activations to", activation_cache_path)

for layer, acts in all_activations.items():
    print(f"{layer}: {acts.shape}")

Extracting activations (this will take a while)...
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 171MB/s]
100%|██████████| 1000/1000 [01:25<00:00, 11.66it/s]


Saved activations to data/processed/alexnet_activations.npz
conv1: (1000, 193600)
conv2: (1000, 139968)
conv3: (1000, 64896)
conv4: (1000, 43264)
conv5: (1000, 43264)
fc6: (1000, 4096)
fc7: (1000, 4096)
fc8: (1000, 1000)


## Sanity check before running any regression

This assert is what would have caught the original bug immediately,
instead of failing three cells later with a cryptic IndexError.

In [9]:
for layer, acts in all_activations.items():
    assert acts.shape[0] == NUM_TRAIN_VIDEOS, (
        f"Layer {layer} has {acts.shape[0]} rows but fMRI data has "
        f"{NUM_TRAIN_VIDEOS} rows. These must match before running regression."
    )

for roi_name, data in roi_data.items():
    assert data.shape[0] == NUM_TRAIN_VIDEOS, (
        f"ROI {roi_name} has {data.shape[0]} rows, expected {NUM_TRAIN_VIDEOS}."
    )

print("All shapes match. Safe to proceed.")

All shapes match. Safe to proceed.


## Train the encoding model: one layer, one region, as a first test

In [10]:
X = all_activations["conv5"]
Y = roi_data["FFA"]

voxel_scores = train_and_evaluate_encoding_model(X, Y)
print("Mean FFA accuracy (conv5):", region_mean_accuracy(voxel_scores))

Mean FFA accuracy (conv5): 0.11187408591428673


## Full sweep: every layer x every region

In [11]:
import pandas as pd

os.makedirs("results/tables/alexnet", exist_ok=True)

results = []

for layer_name, X in all_activations.items():
    for region_name, Y in roi_data.items():
        voxel_scores = train_and_evaluate_encoding_model(X, Y)
        mean_acc = region_mean_accuracy(voxel_scores)
        results.append({"layer": layer_name, "region": region_name, "accuracy": mean_acc})
        print(f"{layer_name} -> {region_name}: {mean_acc:.4f}")

results_df = pd.DataFrame(results)
results_df.to_csv("results/tables/alexnet/baseline_accuracy.csv", index=False)
results_df.head()

conv1 -> V1: 0.0312
conv1 -> V2: 0.0295
conv1 -> V3: 0.0195
conv1 -> V4: 0.0272
conv1 -> LOC: 0.0129
conv1 -> EBA: 0.0117
conv1 -> FFA: 0.0071
conv1 -> STS: -0.0041
conv1 -> PPA: 0.0181


/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.50579e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.90214e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.56285e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.56032e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.51305e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)


conv2 -> V1: 0.0617


/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.50579e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.90214e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.56285e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.56032e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.51305e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)


conv2 -> V2: 0.0666


...

/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.50579e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.90214e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.56285e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.56032e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.51305e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)


conv2 -> STS: 0.0179


/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.50579e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.90214e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.56285e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.56032e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.51306e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)


conv2 -> PPA: 0.0364
conv3 -> V1: 0.0671
conv3 -> V2: 0.0731
conv3 -> V3: 0.0741
conv3 -> V4: 0.0542
conv3 -> LOC: 0.0812
conv3 -> EBA: 0.0854
conv3 -> FFA: 0.0605
conv3 -> STS: 0.0286
conv3 -> PPA: 0.0538
conv4 -> V1: 0.0955
conv4 -> V2: 0.1080
conv4 -> V3: 0.1007
conv4 -> V4: 0.0869
conv4 -> LOC: 0.1052
conv4 -> EBA: 0.1145
conv4 -> FFA: 0.0943
conv4 -> STS: 0.0452
conv4 -> PPA: 0.0815
conv5 -> V1: 0.1111
conv5 -> V2: 0.1254
conv5 -> V3: 0.1185
conv5 -> V4: 0.1132
conv5 -> LOC: 0.1128
conv5 -> EBA: 0.1115
conv5 -> FFA: 0.1119
conv5 -> STS: 0.0471
conv5 -> PPA: 0.0891
fc6 -> V1: 0.0864
fc6 -> V2: 0.1031
fc6 -> V3: 0.0950
fc6 -> V4: 0.0902
fc6 -> LOC: 0.0951
fc6 -> EBA: 0.1016
fc6 -> FFA: 0.0970
fc6 -> STS: 0.0325
fc6 -> PPA: 0.0774
fc7 -> V1: 0.0995
fc7 -> V2: 0.1115
fc7 -> V3: 0.1098
fc7 -> V4: 0.1178
fc7 -> LOC: 0.1297
fc7 -> EBA: 0.1318
fc7 -> FFA: 0.1298
fc7 -> STS: 0.0501
fc7 -> PPA: 0.1153
fc8 -> V1: 0.0578
fc8 -> V2: 0.0887
fc8 -> V3: 0.0936
fc8 -> V4: 0.1000
fc8 -> LOC: 0.1283

,layer,region,accuracy
0,conv1,V1,0.031182
1,conv1,V2,0.029457
2,conv1,V3,0.019530
3,conv1,V4,0.027237
4,conv1,LOC,0.012907


In [12]:
from google.colab import files
files.download("results/tables/alexnet/baseline_accuracy.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

This CSV (`results/tables/alexnet/baseline_accuracy.csv`) is your saved
baseline. Everything in the ablation notebook later gets compared
against these numbers, so keep this file around (commit it to the repo
once you're happy with it).